# Budapest Spa Demo - Semantic View & Cortex Agent

This notebook creates:
1. **Semantic View** for Cortex Analyst (natural language to SQL)
2. **Cortex Agent** for Snowflake Intelligence

In [64]:
import os
from snowflake.snowpark import Session

session = Session.builder.config("connection_name", os.getenv("SNOWFLAKE_CONNECTION_NAME", "oregon_tp")).create()
print(f"Connected as: {session.get_current_user()}")
print(f"Role: {session.get_current_role()}")
print(f"Warehouse: {session.get_current_warehouse()}")

Connected as: "admin"
Role: "ACCOUNTADMIN"
Warehouse: "AI_WH"


In [9]:
session.sql("USE DATABASE BUDAPEST_SPA_DEMO").collect()
session.sql("USE SCHEMA ANALYTICS").collect()
print("Using BUDAPEST_SPA_DEMO.ANALYTICS")

Using BUDAPEST_SPA_DEMO.ANALYTICS


## Step 1: Create Semantic View for Cortex Analyst

In [ ]:
semantic_view_ddl = """
CREATE OR REPLACE SEMANTIC VIEW BUDAPEST_SPA_DEMO.ANALYTICS.SPA_COMPETITIVE_INTEL

TABLES (
    SPAS AS BUDAPEST_SPA_DEMO.ANALYTICS.DIM_SPAS 
        PRIMARY KEY (SPA_ID)
        COMMENT = 'Spa locations - our 3 Aqua Serenity + 32 competitors',
    
    REVIEWS AS BUDAPEST_SPA_DEMO.ANALYTICS.REVIEWS 
        PRIMARY KEY (REVIEW_ID)
        COMMENT = 'Customer reviews with ratings',
    
    SPA_METRICS AS BUDAPEST_SPA_DEMO.ANALYTICS.FACT_SPA_METRICS 
        PRIMARY KEY (SPA_ID, YEAR_MONTH)
        COMMENT = 'Monthly operational metrics'
)

RELATIONSHIPS (
    REVIEWS(SPA_ID) REFERENCES SPAS(SPA_ID),
    SPA_METRICS(SPA_ID) REFERENCES SPAS(SPA_ID)
)

FACTS (
    SPAS.CAPACITY AS CAPACITY COMMENT = 'Max daily visitor capacity',
    SPAS.OPENING_YEAR AS OPENING_YEAR COMMENT = 'Year opened',
    REVIEWS.RATING AS RATING WITH SYNONYMS = ('stars', 'score') COMMENT = 'Star rating 1-5',
    SPA_METRICS.TOTAL_VISITS AS TOTAL_VISITS WITH SYNONYMS = ('visits', 'visitors') COMMENT = 'Monthly visitors',
    SPA_METRICS.REVENUE_HUF AS REVENUE_HUF WITH SYNONYMS = ('revenue', 'sales') COMMENT = 'Revenue in HUF',
    SPA_METRICS.OCCUPANCY_RATE AS OCCUPANCY_RATE WITH SYNONYMS = ('occupancy') COMMENT = 'Capacity utilization'
)

DIMENSIONS (
    SPAS.SPA_ID AS SPA_ID COMMENT = 'Unique spa identifier',
    SPAS.SPA_NAME AS SPA_NAME WITH SYNONYMS = ('spa', 'location', 'name') COMMENT = 'Name of the spa',
    SPAS.DISTRICT AS DISTRICT WITH SYNONYMS = ('area', 'zone') COMMENT = 'Budapest district',
    SPAS.NEIGHBORHOOD AS NEIGHBORHOOD COMMENT = 'Neighborhood name',
    SPAS.SPA_TYPE AS SPA_TYPE WITH SYNONYMS = ('type', 'category') COMMENT = 'Spa type',
    SPAS.PRICE_TIER AS PRICE_TIER WITH SYNONYMS = ('pricing', 'tier') COMMENT = 'Price category',
    SPAS.IS_OWNED AS IS_OWNED WITH SYNONYMS = ('our spa', 'owned', 'aqua serenity', 'ours') COMMENT = 'TRUE = our spa, FALSE = competitor',
    
    REVIEWS.REVIEW_ID AS REVIEW_ID COMMENT = 'Review identifier',
    REVIEWS.REVIEWER_NAME AS REVIEWER_NAME COMMENT = 'Reviewer name',
    REVIEWS.REVIEWER_LOCATION AS REVIEWER_LOCATION WITH SYNONYMS = ('from', 'tourist from') COMMENT = 'Reviewer location',
    REVIEWS.REVIEW_TITLE AS REVIEW_TITLE COMMENT = 'Review title',
    REVIEWS.REVIEW_TEXT AS REVIEW_TEXT WITH SYNONYMS = ('feedback', 'comment') COMMENT = 'Full review content',
    REVIEWS.VISIT_TYPE AS VISIT_TYPE COMMENT = 'Visit type',
    REVIEWS.REVIEW_DATE AS REVIEW_DATE COMMENT = 'Date posted',
    
    SPA_METRICS.YEAR_MONTH AS YEAR_MONTH WITH SYNONYMS = ('month', 'period') COMMENT = 'Month'
)

METRICS (
    REVIEWS.AVG_RATING AS AVG(REVIEWS.RATING) COMMENT = 'Average rating',
    REVIEWS.REVIEW_COUNT AS COUNT(REVIEWS.REVIEW_ID) COMMENT = 'Number of reviews',
    SPA_METRICS.TOTAL_REVENUE AS SUM(SPA_METRICS.REVENUE_HUF) COMMENT = 'Total revenue',
    SPA_METRICS.AVG_VISITS AS AVG(SPA_METRICS.TOTAL_VISITS) COMMENT = 'Average visits'
)

COMMENT = 'Competitive intelligence for Aqua Serenity Spas in Budapest'

AI_SQL_GENERATION 'Our spas have IS_OWNED = TRUE (3 locations). Competitors have IS_OWNED = FALSE (32 locations). Always compare our performance against competitors when relevant.'

AI_QUESTION_CATEGORIZATION '
categorization_rules:
  - name: "employee_data"
    description: "Questions about employee salaries, personal information, or HR data"
    response: "I cannot provide information about employee data. Please contact HR directly."
  - name: "financial_internals"
    description: "Questions about internal profit margins, cost structures, or confidential financial details"
    response: "Detailed financial internals are confidential. I can help with general revenue and performance metrics."
  - name: "competitor_secrets"
    description: "Questions asking for confidential competitor strategies or non-public information"
    response: "I can only provide analysis based on publicly available data and our own operational metrics."
  - name: "personal_reviewer_info"
    description: "Questions trying to identify or contact specific reviewers"
    response: "I cannot provide personal contact information. Reviews are shown anonymously for privacy."
'
"""

# Note: Verified queries are only supported in YAML semantic models, not SQL DDL semantic views.
# To add verified queries, you would need to use a YAML-based semantic model instead.

print(f"Semantic view DDL ready ({len(semantic_view_ddl)} chars)")

Semantic view DDL ready (3170 chars)


In [29]:
session.sql(semantic_view_ddl).collect()
print("✅ Semantic View created!")

✅ Semantic View created!


## Step 2: Test Cortex Analyst

In [31]:
import requests
import json

def ask_analyst(question: str):
    """Call Cortex Analyst REST API to convert natural language to SQL"""
    token = session.connection._rest._token
    account = session.connection.account
    host = session.connection.host
    
    url = f"https://{host}/api/v2/cortex/analyst/message"
    
    headers = {
        "Authorization": f"Snowflake Token=\"{token}\"",
        "Content-Type": "application/json"
    }
    
    payload = {
        "messages": [{"role": "user", "content": [{"type": "text", "text": question}]}],
        "semantic_view": "BUDAPEST_SPA_DEMO.ANALYTICS.SPA_COMPETITIVE_INTEL"
    }
    
    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()
    
    result = response.json()
    return result

def run_analyst_query(question: str):
    """Ask Cortex Analyst and execute the generated SQL"""
    result = ask_analyst(question)
    
    sql = None
    for msg in result.get("message", {}).get("content", []):
        if msg.get("type") == "sql":
            sql = msg.get("statement")
            break
    
    if sql:
        print(f"Generated SQL:\\n{sql}\\n")
        return session.sql(sql).to_pandas()
    else:
        print("Response:", json.dumps(result, indent=2))
        return None

print("✅ Cortex Analyst functions ready")

✅ Cortex Analyst functions ready


In [32]:
result = run_analyst_query("How do our ratings compare to competitors?")
display(result)

Generated SQL:\nSELECT 
  sv.is_owned,
  CASE 
    WHEN sv.is_owned = TRUE THEN 'Our Spas (Aqua Serenity)'
    ELSE 'Competitors'
  END AS spa_category,
  sv.avg_rating
FROM SEMANTIC_VIEW(
  BUDAPEST_SPA_DEMO.ANALYTICS.SPA_COMPETITIVE_INTEL
  METRICS avg_rating
  DIMENSIONS spas.is_owned
) AS sv
ORDER BY sv.is_owned DESC
 -- Generated by Cortex Analyst (request_id: a1271430-1876-496e-9fed-b1b2ed598adc)
;\n


,IS_OWNED,SPA_CATEGORY,AVG_RATING
0,True,Our Spas (Aqua Serenity),3.47619
1,False,Competitors,4.06500


## Step 3: Create Cortex Search Service for Reviews

In [36]:
session.sql("""
CREATE OR REPLACE CORTEX SEARCH SERVICE BUDAPEST_SPA_DEMO.ANALYTICS.SPA_REVIEWS_SEARCH
ON REVIEW_TEXT
ATTRIBUTES SPA_ID, VISIT_TYPE
WAREHOUSE = AI_WH
TARGET_LAG = '1 day'
EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
AS (
    SELECT 
        REVIEW_ID,
        SPA_ID,
        REVIEWER_NAME,
        REVIEWER_LOCATION,
        RATING,
        REVIEW_DATE,
        REVIEW_TITLE,
        REVIEW_TEXT,
        VISIT_TYPE
    FROM BUDAPEST_SPA_DEMO.ANALYTICS.REVIEWS
)
""").collect()
print("✅ Cortex Search Service created!")

✅ Cortex Search Service created!


In [38]:
from snowflake.core import Root

owned_spas = session.sql("""
    SELECT SPA_ID, SPA_NAME FROM BUDAPEST_SPA_DEMO.ANALYTICS.DIM_SPAS 
    WHERE IS_OWNED = TRUE
""").collect()
owned_spa_ids = [row['SPA_ID'] for row in owned_spas]
print(f"Our spas: {[row['SPA_NAME'] for row in owned_spas]}")
print(f"SPA_IDs: {owned_spa_ids}\n")

root = Root(session)
search_service = (root
    .databases["BUDAPEST_SPA_DEMO"]
    .schemas["ANALYTICS"]
    .cortex_search_services["SPA_REVIEWS_SEARCH"]
)

filter_conditions = {"@or": [{"@eq": {"SPA_ID": spa_id}} for spa_id in owned_spa_ids]}

results = search_service.search(
    query="complaints about wait times and crowded",
    columns=["REVIEW_TEXT", "RATING", "REVIEWER_NAME", "REVIEW_TITLE", "SPA_ID"],
    filter=filter_conditions,
    limit=5
)

print("🔍 Search: 'complaints about wait times and crowded' (OUR SPAS ONLY)\n")
for r in results.results:
    print(f"⭐ {r['RATING']} - {r['REVIEW_TITLE']} (Spa ID: {r['SPA_ID']})")
    print(f"   {r['REVIEW_TEXT'][:200]}...")
    print()

Our spas: ['Aqua Serenity Castle District', 'Aqua Serenity City Park', 'Aqua Serenity Buda Hills']
SPA_IDs: [1, 2, 3]

🔍 Search: 'complaints about wait times and crowded' (OUR SPAS ONLY)

⭐ 3 - Good Experience But Overpriced for What You Get (Spa ID: 3)
   **Decent spa but expected more for the price**

My partner and I visited during our Budapest weekend getaway. The location in Buda Hills is lovely and peaceful, away from tourist crowds. Staff were ge...

⭐ 3 - Good Spa Experience But Overpriced for What Offered (Spa ID: 3)
   # Decent spa but not worth the luxury price tag

Visited during my solo trip to Budapest and had mixed feelings. The location in Buda Hills is lovely and peaceful, away from the tourist chaos. Staff w...

⭐ 3 - Relaxing Atmosphere But Overpriced for What You Get (Spa ID: 1)
   **Good location, but needs improvement for premium pricing**

Visited during a solo weekend trip from Munich. The Castle District location is unbeatable – walking distance from major sigh

## Step 4: Create Cortex Agent

In [84]:
session.sql("""
CREATE OR REPLACE AGENT BUDAPEST_SPA_DEMO.ANALYTICS.SPA_COMPETITIVE_ANALYST
  COMMENT = 'Competitive intelligence analyst for Aqua Serenity Spas in Budapest'
  PROFILE = '{
    "display_name": "Spa Competitive Analyst"
  }'
  FROM SPECIFICATION $$
  {
    "models": {
      "orchestration": "claude-sonnet-4-5"
    },
    "instructions": {
      "orchestration": "You help analyze spa competitive intelligence. Use the analyst tool for data queries (ratings, metrics, comparisons). Use the search tool to find specific reviews or customer feedback. Our spas have IS_OWNED=TRUE, competitors have IS_OWNED=FALSE.",
      "response": "Be concise and data-driven. When discussing reviews, include relevant quotes."
    },
    "tools": [
      {
        "tool_spec": {
          "type": "cortex_analyst_text_to_sql",
          "name": "spa_analyst",
          "description": "Query spa competitive intelligence data including ratings, metrics, and aggregated statistics"
        }
      },
      {
        "tool_spec": {
          "type": "cortex_search",
          "name": "review_search",
          "description": "Search customer reviews to find specific feedback, complaints, or praise about spa experiences"
        }
      }
    ],
    "tool_resources": {
      "spa_analyst": {
        "semantic_view": "BUDAPEST_SPA_DEMO.ANALYTICS.SPA_COMPETITIVE_INTEL",
        "execution_environment": {
          "type": "warehouse",
          "warehouse": "AI_WH"
        }
      },
      "review_search": {
        "search_service": "BUDAPEST_SPA_DEMO.ANALYTICS.SPA_REVIEWS_SEARCH",
        "max_results": 10
      }
    }
  }
  $$
""").collect()
print("✅ Cortex Agent created with Analyst + Search tools!")

✅ Cortex Agent created with Analyst + Search tools!


In [76]:
session.sql("GRANT USAGE ON AGENT BUDAPEST_SPA_DEMO.ANALYTICS.SPA_COMPETITIVE_ANALYST TO ROLE PUBLIC").collect()

session.sql("""
ALTER SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT 
ADD AGENT BUDAPEST_SPA_DEMO.ANALYTICS.SPA_COMPETITIVE_ANALYST
""").collect()

print("✅ Agent added to Snowflake Intelligence!")
print()
print("Access at: https://ai.snowflake.com")

SnowparkSQLException: (1304): 01c2c8a5-0307-f5b0-0041-b7870d214406: 400203 (23505): SPA_COMPETITIVE_ANALYST is already present in SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT.

## Step 5: Test Agent

In [85]:
def ask_agent(question: str):
    """Call Cortex Agent via SQL function"""
    row = session.sql(f"""
    SELECT TRY_PARSE_JSON(
        SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
            'BUDAPEST_SPA_DEMO.ANALYTICS.SPA_COMPETITIVE_ANALYST',
            $${{
                "messages": [
                    {{
                        "role": "user",
                        "content": [
                            {{"type": "text", "text": "{question}"}}
                        ]
                    }}
                ]
            }}$$
        )
    ) AS resp
    """).collect()[0]['RESP']
    
    # Parse if it's still a string
    if isinstance(row, str):
        result = json.loads(row)
    else:
        result = row
    
    # Extract text response from content array
    if result and isinstance(result, dict) and 'content' in result:
        for item in result['content']:
            if isinstance(item, dict) and item.get('type') == 'text':
                return {"response": item.get('text', ''), "raw": result}
    
    return {"response": "", "raw": result}

print("Function defined - run cell 17 to test")

Function defined - run cell 17 to test


In [86]:
result = ask_agent("How do our ratings compare to competitors?")
print("Response:", result.get('response', '')[:500])
print("\nFull result:")
print(json.dumps(result.get('raw'), indent=2, default=str))

Response: Our spas (Aqua Serenity) have an average rating of **3.48** based on 21 reviews, while competitors have an average rating of **4.07** based on 200 reviews. Competitors currently have a higher average rating by approximately 0.59 points.



Full result:
{
  "content": [
    {
      "thinking": {
        "text": "This question looks similar to a verified analyst query. Let me try to use the analyst tool."
      },
      "type": "thinking"
    },
    {
      "tool_use": {
        "client_side_execute": false,
        "input": {
          "previous_related_tool_result_id": "",
          "query": "How do our ratings compare to competitors?",
          "reference_vqrs": [
            {
              "confidence": 1,
              "name": "\"How do our ratings compare to competitors?\"",
              "question": "How do our ratings compare to competitors?",
              "sql": "SELECT\n  sv.is_owned,\n  CASE\n    WHEN sv.is_owned = TRUE THEN 'Our Spas (Aqua Serenity)'\n    ELSE 'C

In [ ]:
test_questions = [
    # Andras (CEO)
    "How do our ratings compare to the top 5 competitors?",
    "Which of our locations is underperforming and why?",
    "What would it take to reach a 4.5 average rating?",
    # Eva (CMO)
    "Which districts have no premium spas? That's our opportunity.",
    "What do reviews mention that we're not doing?",
    # Peter (Operations)
    "What are the top 3 complaints at each of our locations?",
    "Which treatments should we add based on competitor success?",
    # Agent Testing
    "How are we doing compared to Szechenyi Baths?",
    "What should we fix at our Downtown location?",
    "Where should we open our 4th spa?"]